In data analysis and modeling, data preparation load, clean, transform, rearrange takes up much time (80%+). How data is stored in files/ DBs might be in the wrong format for some task. Many researchers do ad hoc data processing using a general-purpose language (Python, Perl, R, Java, or Unix text-processing tools: sed/ awk). pd and built-in Python features give high-level, flexible, and fast tools to turn data into the right form.<br>
If you find a type of data manipulation that is not in this book or pd, share your use case on a Python mailing lists or the pd GitHub site. Real-world application needs have often driven pd design and implementation.<br>
This chapter covers tools for missing data, duplicate data, string manipulation, and other analytical data transformations. The next combines and rearranges datasets.

In [ ]:
import numpy as np
import pandas as pd
import re

Missing data is common. pd tries to make working with it painless, eg descriptive statistics on pd objects exclude it by default.
How pd objects represent it is imperfect, but suffices for most real-world use. For <i>dtype=float64</i>, pd uses the floating-point value NaN (Not a Number).
<i>None</i> is treated as null too. In general, a sentinel value indicates a missing (null) value

In [ ]:
float_data = pd.Series([1.2, -3.5, np.nan, 0, None])
float_data.isna() #recall this is 1 way to check for null values

pd adopts a convention from R to refer to missing data as NA. In stats applications, NA data may be data that does not exist or that exists but was not observed (eg problems with data collection). When data cleaning, it is important to do analysis on NAs to identify data collection problems or potential biases it causes

In [ ]:
string_data = pd.Series(["aardvark", np.nan, None, "avocado"])
string_data.isna()

Methods to handle NA
<table>
<tr><th>Method</th><th>Description</th></tr>
<tr><td>dropna</td>
<td>Filter axis labels based on whether values for each label have missing data,<br> with varying thresholds for how much missing data to tolerate.</td></tr>
<tr><td>fillna</td>
<td>Fill in missing data with some value or using an interpolation method such as "ffill" or "bfill".</td></tr>
<tr><td>isna</td>
<td>Return Boolean values indicating which values are missing/NA.</td></tr>
<tr><td>notna</td>
<td>Negation of isna, returns True for non-NA values and False for NA values.</td></tr>
</table>

To filter out NAs, use <i>isna()</i> and Boolean indexing or <i>dropna()</i>, which returns a copy. For <i>pd.Series</i>, it just keeps the nonnull data and index values

In [ ]:
data = pd.Series([1, None, 3.5, np.nan, 7])
data.dropna()
data[data.notna()] #same

In [ ]:
df = pd.DataFrame(np.random.standard_normal((7, 3)))
df.iloc[6, 0] = None
df.iloc[5:, 1] = np.nan
df.iloc[3:, 2] = np.nan
df
df.dropna() #drop rows with any NA
df.dropna(how="all") #drop rows with all NA
df.dropna(thresh=2) #drop rows with 2+ NA
df.dropna(axis=1,thresh=5) #drop columns with <5 non NA

Besides filtering out NAs (and likely discarding other data too), you can fill in the 'holes'. <i>fillna()<i> is the workhorse method. Calling it with a constant replaces NAs with that. Calling it with a dict allows a different fill value per column (each key must match a column name)

In [ ]:
df.fillna(0)
df.fillna({1: 0.5, 2: 0})
df.ffill() #forward fill
df.ffill(limit=2) #forward fill up to 2 consecutive NAs
df.fillna(df.mean()) #impute column means

The DataFrame <i>duplicated()</i> method returns a Boolean pd.Series indicating if each row is a duplicate (its column values are exactly equal to those in an earlier row, use keep="last" for a later row). Relatedly, <i>drop_duplicates()</i>  returns a DataFrame with those rows removed. Both by default consider all columns. Alternatively, use a <i>subset</i> of them to detect duplicates.

In [ ]:
data = pd.DataFrame({"k1": ["one", "two"] * 3 + ["two"],
                     "k2": [1, 1, 2, 3, 3, 4, 4]})
data.duplicated()
data.drop_duplicates()
data["v1"] = range(7)
data.drop_duplicates(subset="k1")
data.drop_duplicates(["k1", "k2"], keep="last")

Recall (Ch 5) the Series <i>map()</i> method transforms data and accepts a dict-like object or function

In [ ]:
data = pd.DataFrame({"food": ["bacon", "pulled pork", "bacon",
                              "pastrami", "corned beef", "bacon",
                              "pastrami", "honey ham", "nova lox"],
                     "ounces": [4, 3, 12, 6, 7.5, 8, 3, 5, 6]})
meat_to_animal = {"bacon": "pig","pulled pork": "pig",
  "pastrami": "cow","corned beef": "cow",
  "honey ham": "pig","nova lox": "salmon"}
def get_animal(x):
    return meat_to_animal[x]
data["animal"] = data["food"].map(meat_to_animal)
data["food"].map(get_animal) #same

The <i>data.replace()</i> method is distinct from <i>data.str.replace()</i>, which performs element-wise string substitution. We further study these string methods later.

In [ ]:
data = pd.Series([1., -999., 2., -999., -1000., 3.])
data.replace(-999, np.nan) #assume -999 is a sentinel for NA
data.replace([-999, -1000], np.nan)
data.replace([-999, -1000], [np.nan, 0])
data.replace({-999: np.nan, -1000: 0}) #same via dict

In [ ]:
#map is 1 way to rename axes
data = pd.DataFrame(np.arange(12).reshape((3, 4)),
                    index=["Ohio", "Colorado", "New York"],
                    columns=["one", "two", "three", "four"])
def transform(x):
    return x[:4].upper()
data.index.map(transform) #data unchanged
data.index = data.index.map(transform)

<i>rename()</i> renames the index and/ or columns via a dict- like or function. It saves you from manually assigning new values to the index/ columns

In [ ]:
data.rename(index=str.title, columns=str.upper)
data.rename(index={"OHIO": "INDIANA"},columns={"three": "peekaboo"})

<i>pd.cut()</i> discretizes continuous data based on <i>bins</i>, an iterable of cutoffs (default), or an int to use equally spaced intervals (use <i>precision</i> for how many decimal places in their endpoints). The output is a special Categorical object. A special (unique to pd) interval value type with the lower (inclusive) and upper (exclusive) limit identifies each bin. To make the right endpoints inclusive (and left exclusive), use <i>right=False</i>. Use <i>labels</i> to override the default interval based labeling

In [ ]:
ages = [20, 22, 25, 27, 21, 23, 37, 31, 61, 45, 41, 32]
bins = [18, 25, 35, 60, 100]
age_categories = pd.cut(ages, bins=bins)
age_categories.codes #np.ndarray of 0,...,n_bins-1
age_categories.categories
#pd.value_counts(age_categories)
age_categories.value_counts()

In [ ]:
pd.cut(ages, bins, right=False)
pd.cut(ages, bins, labels=["Youth", "YoungAdult", "MiddleAged", "Senior"])
pd.cut(ages, bins=4, precision=2)

Closely related, <i>pd.qcut()</i> discretizes based on <i>q</i>, the number (each bin will have roughly the same number of data points) of sample quantiles, or explicitly specify them. These 2 binning functions appear again later when we aggregate and do group operations.

In [ ]:
pd.qcut(ages, q=4, precision=2).value_counts()
pd.qcut(ages, q=[0, 0.1, 0.5, 0.9, 1.]).value_counts()

To find/ filter/ transform outliers, apply array operations. Below we do it for 3 SDs for normally distributed data, truncate values beyond 3 in abs value to +/- 3 depending on sign

In [ ]:
data = pd.DataFrame(np.random.standard_normal((1000, 4)))
data.describe()
col = data[2]
col[col.abs() > 3] #find outliers in a series
data[(data.abs() > 3).any(axis=1)] #rows with any outlier
data[data.abs() > 3] = np.sign(data) * 3 #truncate
data.describe()

In [ ]:
#permute and/ or randomly sample rows/ columns
df = pd.DataFrame(np.arange(5 * 7).reshape((5, 7)))
rng = np.random.default_rng()
sampler = rng.permutation(5)
df.iloc[sampler]
column_sampler = rng.permutation(7)
df.iloc[:, column_sampler]
df.sample(n=3) #rows without replacement
df.sample(n=8, replace=True, axis=1) #columns with replacement

In [ ]:
#A common stats modeling/ ML transform is one hot encoding
df = pd.DataFrame({"key": ["b", "b", "a", "c", "a", "b"],
                   "key2": ["b", "b", "a", "c", "a", None],
                   "data1": range(6)})
pd.get_dummies(df["key"]) #3 new boolean columns
pd.get_dummies(df["key"],drop_first=True)
pd.get_dummies(df["key"],dtype=int)
pd.get_dummies(df["key"], prefix="key") #customize new column names
pd.get_dummies(df, columns=["key"]) #keep other columns in df

To binarize multi-label strings (eg movie genres separated by |), use <i>str.get_dummies()</i> (unlike <i>pd.get_dummies()</i>, its default output type is int64, ie 0/1. It also lacks a prefix argument, so use the <i>add_prefix()</i> method for the same functionality). To keep the other columns, merge the result back in via the <i>join()</i> method (more on merging later).

In [ ]:
movies = pd.read_table("https://raw.githubusercontent.com/wesm/pydata-book/refs/heads/3rd-edition/datasets/movielens/movies.dat",
  sep="::",index_col=0,header=None, names=["title", "genres"], engine="python")
movies.head()
dummies = movies["genres"].str.get_dummies("|")
dummies.head().iloc[:, :6]
movies_windic = movies.join(dummies.add_prefix("Genre_"))
movies_windic.iloc[0]

<i>MultiLabelBinarizer</i> in sklearn is a faster version of <i>str.get_dummies()</i>. Below we use %timeit to time a single line, and %%timeit for a whole cell

In [ ]:
%timeit movies["genres"].str.get_dummies("|")

In [ ]:
%%timeit
#from sklearn.preprocessing import MultiLabelBinarizer
genres_list = movies['genres'].str.split('|').tolist()
mlb = MultiLabelBinarizer()
dummies_skl = pd.DataFrame(mlb.fit_transform(genres_list),
                           columns=[f"Genre_{g}" for g in mlb.classes_],
                           index=movies.index)
#movies_skl = movies.join(dummies_skl)

In [ ]:
dummies.add_prefix("Genre_").equals(dummies_skl)

In [ ]:
# often combine get_dummies with pd.cut
values = np.random.uniform(size=10)
bins = [0, 0.2, 0.4, 0.6, 0.8, 1]
pd.get_dummies(pd.cut(values, bins))

Extension data types is a newer and more advanced topic. pd was originally built upon np capabilities while trying to maximize compatibility between libraries that used both NumPy and pandas.<br>
Building on NumPy led to shortcomings. When missing data was introduced into some data types (eg integers and Booleans), pd converted to float64 and used np.nan to represent NAs. This had compounding effects by introducing subtle issues into many pd algorithms. Datasets heavy on string data were computationally expensive and used a lot of memory. Some data types (eg intervals, timedeltas, and timestamps with time zones) could not be supported efficiently without using computationally expensive arrays of Python objects.<br>
More recently, pd developed an extension type system that lets new data types be added even if not supported natively by np. They can be treated as first class alongside data coming from np arrays.

For backward compatibility, pd.Series with ints becomes float64 in the presence of NAs (recall they are np.nan). pd thus made pd.Int64Dtype(), or Int64 for short (note the capital I). Now NA is represented by pd.NA, distinct and displays differently from np.nan

In [ ]:
pd.Series([1, 2, 3]) #dtype int64
pd.Series([1, 2, 3, None]) # dtype float64
s = pd.Series([1, 2, 3, None], dtype="Int64") #same dtype=pd.Int64Dtype()
s[3] is pd.NA
s.isna() #still detected
np.nan, pd.NA #display differently

Using pd.StringDtype() (need the pyarrow module) avoids np object arrays. It uses less memory and is moore efficient computationally. Another extension we use later is pd.CategoricalDtype() (category for short). Convert a pd.Series dtype via the <i>astype()</i> method

In [ ]:
s = pd.Series(['one', 'two', None, 'three'], dtype='string') #same pd.StringDtype()
df = pd.DataFrame({"A": [1, 2, None, 4],
                   "B": ["one", "two", "three", None],
                   "C": [False, None, False, True]})
df["A"] = df["A"].astype("Int64")
df["B"] = df["B"].astype("string")
df["C"] = df["C"].astype("boolean")
df.dtypes

<table>
<tr><td>Extension type</td><td>Description</td><td>Shorthand</td></tr>
<tr><td>BooleanDtype</td><td>Nullable Boolean data</td><td>boolean</td></tr>
<tr><td>CategoricalDtype</td><td>Categorical data type</td><td>category</td></tr>
<tr><td>DatetimeTZDtype</td><td>Datetime with time zone</td><td></td></tr>
<tr><td>Float32Dtype</td><td>32-bit nullable floating point</td><td>Float32</td></tr>
<tr><td>Float64Dtype</td><td>64-bit nullable floating point</td><td>Float64</td></tr>
<tr><td>Int8Dtype</td><td>8-bit nullable signed integer</td><td>Int8</td></tr>
<tr><td>Int16Dtype</td><td>16-bit nullable signed integer</td><td>Int16</td></tr>
<tr><td>Int32Dtype</td><td>32-bit nullable signed integer</td><td>Int32</td></tr>
<tr><td>Int64Dtype</td><td>64-bit nullable signed integer</td><td>Int64</td></tr>
<tr><td>UInt8Dtype</td><td>8-bit nullable unsigned integer</td><td>UInt8</td></tr>
<tr><td>UInt16Dtype</td><td>16-bit nullable unsigned integer</td><td>UInt16</td></tr>
<tr><td>UInt32Dtype</td><td>32-bit nullable unsigned integer</td><td>UInt32</td></tr>
<tr><td>UInt64Dtype</td><td>64-bit nullable unsigned integer</td><td>UInt64</td></tr>
</table>

Python has string and text processing tools. The str object's built-in methods handle most text operations. More complex pattern matching and text manipulations may be need regular expressions. pd enables applying string and regular expressions concisely on whole arrays of data, and handles the annoyance of NAs


In [ ]:
#break a str into pieces
val = "a,b,  guido"
val.split(sep=",")

<i>split()</i> is often combined with <i>strip()</i>, which trims whitespace, including line breaks. Call <i>join()</i> on a separator str, with an iterable of strings as input, to concatenate them

In [ ]:
pieces = [x.strip() for x in val.split(",")]
"::".join(pieces)
"".join(pieces)

Recall the <i>in</i> operator detects substrings. To get the position, use <i>find()</i> (return -1 if not found) or <i>index()</i> (return ValueError if not found), which are otherwise identical. Recall <i>count()</i> gets how many times a substring appears, <i>replace()</i> replaces a pattern with another. Use '' to delete occurrences

In [ ]:
"guido" in val
val.find(",")
val.index(",") #same
val.find(":") #-1
#val.index(":") #ValueError
val.count(",")
val.replace(",", "::")
val.replace(",", "") #delete all ','

str methods
<table border="1" cellpadding="4" cellspacing="0">
<tr><th>Method</th>
<th>Description</th></tr>
<tr><td>count</td>
<td>Return the number of nonoverlapping occurrences of substring in the string</td></tr>
<tr><td>endswith</td>
<td>Return True if string ends with suffix</td></tr>
<tr><td>startswith</td>
<td>Return True if string starts with prefix</td></tr>
<tr><td>join</td>
<td>Use string as delimiter for concatenating a sequence of other strings</td></tr>
<tr><td>index</td>
<td>Return starting index of the first occurrence of passed substring if found in the string;<br> otherwise, raises ValueError if not found</td></tr>
<tr><td>find</td>
<td>Return position of first character of first occurrence of substring in the string;<br> like index, but returns –1 if not found</td></tr>
<tr><td>rfind</td>
<td>Return position of first character of last occurrence of substring in the string;<br> returns –1 if not found</td></tr>
<tr><td>replace</td>
<td>Replace occurrences of string with another string</td></tr>
<tr><td>strip, rstrip, lstrip</td>
<td>Trim whitespace, including newlines on both sides, on the right side,<br> or on the left side, respectively</td></tr>
<tr><td>split</td>
<td>Break string into list of substrings using passed delimiter</td></tr>
<tr><td>lower</td>
<td>Convert alphabet characters to lowercase</td></tr>
<tr><td>upper</td>
<td>Convert alphabet characters to uppercase</td></tr>
<tr><td>casefold</td>
<td>Convert characters to lowercase, and convert any region-specific variable character combinations<br> to a common comparable form</td></tr>
<tr><td>ljust, rjust</td>
<td>Left justify or right justify, respectively; pad opposite side of string with spaces<br> (or some other fill character) to return a string with a minimum width</td></tr>
</table>

Regular expressions are more flexible and often complex to handle strings, usually to match patterns, substitute, or split. Use Python's <i>re</i> module. A single such expression is usually called a regex.<br>
As a 1st example, r"\s+" is a regex to detect any nonzero amount of whitespace (including tab, newline). '\\s+' works too. Recall the r stands for raw. Without it, python processes backslashes before the regex engine sees them. We use it to split a str, which <i>str.split()</i> does by default

In [ ]:
text = "foo    bar\t baz  \tqux"
re.split(r"\s+", text)==text.split() #True
re.split("\\s+", text) #same

When we call <i>re.split()</i>, the regex is 1st compiled, then its <i>split()</i> method is called on the passed text. We do this manually, giving a reusable regex object (saves CPU cycles if you need to use it many times).

In [ ]:
regex = re.compile(r"\s+")
regex.split(text)

<i>findall()</i> gives a list of all (maximal) substrings matching the regex. Relatedly, <i>search()</i> returns only the first match. More rigidly, <i>match()</i> only matches if the pattern occurs at the string's start. <i>flags=re.IGNORECASE</i> makes the regex case insensitive. Both the latter 2 functions give a <i>re.Match</i> object. <i>sub()</i> gives new string with occurrences of the pattern replaced by the 1st argument

In [ ]:
regex.findall(text)
re.findall(r"\s+", text) #same

In [ ]:
text = """Dave dave@google.com
Steve steve@gmail.com
Rob rob@gmail.com
Ryan ryan@yahoo.com"""
pattern = r"[A-Z0-9._%+-]+@[A-Z0-9.-]+\.[A-Z]{2,4}" #match an email

regex = re.compile(pattern, flags=re.IGNORECASE)
regex.findall(text)
m = regex.search(text)
re.search(pattern, text, flags=re.IGNORECASE) #same, but not ==m
m==regex.search(text) #False
type(m) #re.Match
text[m.start():m.end()] #only works if something was found
m=regex.match(text) #None since text does not start with an email
#text[m.start():m.end()] #AttributeError
regex.sub("REDACTED", text)

To find email addresses and segment each them into 3 components (username, domain name, domain suffix), put parentheses around those parts of the pattern. The resulting <i>re.Match</i> object has a tuple of those parts in its <i>groups()</i> method. <i>findall()</i> gives a list of tuples, 1 per match. <i>sub()</i> can also access these pieces using <i>\1, \2</i> etc

In [ ]:
pattern = r"([A-Z0-9._%+-]+)@([A-Z0-9.-]+)\.([A-Z]{2,4})"
regex = re.compile(pattern, flags=re.IGNORECASE)
m = regex.match("wesm@bright.net 368@qjx.edu")
m.groups()
regex.findall(text)
regex.sub(r"Username: \1, Domain: \2, Suffix: \3", text)

<table>
<tr><th>Method</th>
<th>Description</th></tr>
<tr><td>findall</td>
<td>Return all nonoverlapping matching patterns in a string as a list</td></tr>
<tr><td>finditer</td>
<td>Like findall, but returns an iterator</td></tr>
<tr><td>match</td>
<td>Match pattern at start of string and optionally segment pattern<br>components into groups; if the pattern matches, return a match object,<br>and otherwise None</td></tr>
<tr><td>search</td>
<td>Scan string for match to pattern, returning a match object if so;<br>unlike match, the match can be anywhere in the string as opposed to<br>only at the beginning</td></tr>
<tr><td>split</td>
<td>Break string into pieces at each occurrence of pattern</td></tr>
<tr><td>sub, subn</td>
<td>Replace all (sub) or first n occurrences (subn) of pattern in string<br>with replacement expression; use symbols \1, \2, ... to refer to<br>match group elements in the replacement string</td></tr>
</table>

Cleaning data often requires string manipulation. To complicate matters, a column with strings can have NAs. String and regular expression methods can be applied (eg, pass a lambda) via the <i>map()</i> method, but it fails on NAs. To cope, pd.Series has array-oriented methods (accessed via the <i>str</i> attribute) to skip over and propagate NAs. <i>str.contains()</i> is an analog for the <i>in</i> operator

In [ ]:
data = {"Dave": "dave@google.com", "Steve": "steve@gmail.com",
        "Rob": "rob@gmail.com", "Wes": np.nan}
data = pd.Series(data)
data.dropna().map(lambda q: 'gmail' in q) #dtype bool
data.dropna().str.contains("gmail") #same
#data.map(lambda q: 'gmail' in q) #Type error since has NA
data.str.contains("gmail") #dtype object

If we 1st convert to extension dtype <i>string</i>, then <i>str.contains()</i> outputs as extension dtype <i>boolean</i>

In [ ]:
data_as_string_ext = data.astype('string')
data_as_string_ext
data_as_string_ext.str.contains("gmail")

<i>str.findall()</i> is the analog of <i>re.findall()</i>. Per pd.Series entry, give a list of matches. If <i>pat</i> has capture groups, include a tuple of them per match

In [ ]:
pattern0 = r"[A-Z0-9._%+-]+@[A-Z0-9.-]+\.[A-Z]{2,4}"
pattern = r"([A-Z0-9._%+-]+)@([A-Z0-9.-]+)\.([A-Z]{2,4})"
data.str.findall(pat=pattern0, flags=re.IGNORECASE)
data.str.findall(pattern, flags=re.IGNORECASE)

To do vectorized element retrieval, use <i>str.get()</i> or index into the <i>str</i> attribute. The latter allows slicing too. <i>str.extract()</i> returns a regex's captured groups as a pd.DataFrame

In [ ]:
matches = data.str.findall(pattern, flags=re.IGNORECASE).str[0]
#matches = data.str.findall(pattern, flags=re.IGNORECASE).str.get(0) #same
matches
matches.str.get(1) #same matches.str[1]
data.str[:5]
data.str.extract(pattern, flags=re.IGNORECASE)

<table>
<tr><th>Method</th>
<th>Description</th></tr>
<tr><td>cat</td>
<td>Concatenate strings element-wise with optional delimiter</td></tr>
<tr><td>contains</td>
<td>Return Boolean array if each string contains pattern/regex</td></tr>
<tr><td>count</td>
<td>Count occurrences of pattern</td></tr>
<tr><td>extract</td>
<td>Use a regular expression with groups to extract one or more strings<br>from a Series of strings; the result will be a DataFrame with one<br>column per group</td></tr>
<tr><td>endswith</td>
<td>Equivalent to x.endswith(pattern) for each element</td></tr>
<tr><td>startswith</td>
<td>Equivalent to x.startswith(pattern) for each element</td></tr>
<tr><td>findall</td>
<td>Compute list of all occurrences of pattern/regex for each string</td></tr>
<tr><td>get</td>
<td>Index into each element (retrieve i-th element)</td></tr>
<tr><td>isalnum</td>
<td>Equivalent to built-in str.alnum</td></tr>
<tr><td>isalpha</td>
<td>Equivalent to built-in str.isalpha</td></tr>
<tr><td>isdecimal</td>
<td>Equivalent to built-in str.isdecimal</td></tr>
<tr><td>isdigit</td>
<td>Equivalent to built-in str.isdigit</td></tr>
<tr><td>islower</td>
<td>Equivalent to built-in str.islower</td></tr>
<tr><td>isnumeric</td>
<td>Equivalent to built-in str.isnumeric</td></tr>
<tr><td>isupper</td>
<td>Equivalent to built-in str.isupper</td></tr>
<tr><td>join</td>
<td>Join strings in each element of the Series with passed separator</td></tr>
<tr><td>len</td>
<td>Compute length of each string</td></tr>
<tr><td>lower, upper</td>
<td>Convert cases; equivalent to x.lower() or x.upper() for each element</td></tr>
<tr><td>match</td>
<td>Use re.match with the passed regular expression on each element,<br>returning True or False whether it matches</td></tr>
<tr><td>pad</td>
<td>Add whitespace to left, right, or both sides of strings</td></tr>
<tr><td>center</td>
<td>Equivalent to pad(side="both")</td></tr>
<tr><td>repeat</td>
<td>Duplicate values (e.g., s.str.repeat(3) is equivalent to x * 3 for<br>each string)</td></tr>
<tr><td>replace</td>
<td>Replace occurrences of pattern/regex with some other string</td></tr>
<tr><td>slice</td>
<td>Slice each string in the Series</td></tr>
<tr><td>split</td>
<td>Split strings on delimiter or regular expression</td></tr>
<tr><td>strip</td>
<td>Trim whitespace from both sides, including newlines</td></tr>
<tr><td>rstrip</td>
<td>Trim whitespace on right side</td></tr>
<tr><td>lstrip</td>
<td>Trim whitespace on left side</td></tr>
</table>

We now introduce pd's <i>category</i> extension dtype, which can gives better performance and memory use. A table column often has a small set of unique values with many repeats. Recall <i>unique()</i> extracts distinct values and <i>value_counts()</i> finds their frequencies

In [ ]:
values = pd.Series(['apple', 'orange', 'apple','apple'] * 2)
pd.unique(values)
values.value_counts()

Many data systems (eg for data warehousing or stats computing) have specialized ways to represent data with repeated values for more efficient storage and computation. In data warehousing, a best practice uses dimension tables with the distinct values and stores the primary observations as integer keys referencing those tables. <i>take()</i> or fancy indexing restores the original pd.Series of strings. This is called the categorical or dictionary-encoded representation. The array of distinct values is called the categories, dictionary, or levels of the data.<br>
This representation improves performance for analytics. You can transform the categories while leaving the codes unmodified. Some low cost examples are: rename categories, append a new category without changing the order/ position of existing.<br>
All our examples use strings, but categories can be any immutable values

In [ ]:
values = pd.Series([0, 1, 0, 0] * 2)
dim = pd.Series(['apple', 'orange'])
dim.iloc[values] #dim[values] or dim.take(values)

<i>astype()</i> converts to  pd's <i>category</i> extension dtype that uses the integer-based categorical encoding

In [ ]:
rng = np.random.default_rng()
df = pd.DataFrame({'fruit': ['apple', 'orange', 'apple', 'apple'] * 2,
                   #'basket_id': np.arange(8),
                   'count': rng.integers(3, 15, size=8),
                   'weight': rng.uniform(0, 4, size=8)})
df['fruit'] = df['fruit'].astype('category')

The <i>array</i> attribute of a pd.Series with <i>category</i> dtype can be made directly via the <i>pd.Categorical</i> class.

In [ ]:
type(df['fruit'].array)==pd.Categorical #True
qjx = pd.Categorical(['foo', 'bar', 'baz', 'foo', 'bar'])
qjx.categories #type pd.Index
qjx.codes #type np.ndarray
dict(enumerate(qjx.categories)) #see map between codes and categories

You can start with encoded data and use the alternative <i>from_codes()</i> constructor to make a pd.Categorical array. No specific category ordering is assumed by default. When using any constructor, set <i>ordered=True</i> to put them in the same order as <i>categories</i>.

In [ ]:
categories = ['foo', 'bar', 'baz']
codes = [0, 1, 2, 0, 0, 1]
my_cats_2 = pd.Categorical.from_codes(codes, categories=categories)
ordered_cat = pd.Categorical.from_codes(codes, categories,ordered=True)
my_cats_2.as_ordered() #convert to ordered

The <i>category</i> dtype in pd compared with the nonencoded version (eg a string array) mostly behaves the same way. Some methods (eg <i>groupby()</i>) perform better. Some have an <i>ordered</i> flag. We revisit <i>pd.(q)cut()</i>. Its output is of class <i>pd.Categorical</i>. We use <i>groupby()</i> to extract summary stats. Each unique value of the grouping variable appears in the 1st column

In [ ]:
#rng = np.random.default_rng()
draws = rng.standard_normal(36)
bins = pd.qcut(draws, 4, labels=['Q1', 'Q2', 'Q3', 'Q4'])
type(bins)==pd.Categorical #True

In [ ]:
bins = pd.Series(bins, name='quartile')
results = (pd.Series(draws)
           .groupby(bins)
           .agg(['count', 'min', 'max'])
           .reset_index())
results

In [ ]:
N = 10_000_000
labels = pd.Series(['foo', 'bar', 'baz', 'qux'] * (N // 4))
categories = labels.astype('category')

In [ ]:
#one-time cost to convert to categorical
%time _ = labels.astype('category')

In [ ]:
labels.memory_usage(deep=True)
labels.info() #also shows memory usage
categories.memory_usage(deep=True)
categories.info()

In [ ]:
%timeit labels.value_counts()
%timeit categories.value_counts()

pd.Series of category dtype have special methods similar to the <i>pd.Series.str</i> specialized string methods, accessed via the special accessor attribute <i>cat</i>. This is another way to access the categories and codes.

In [ ]:
cat_s = pd.Series(['a', 'b', 'c', 'd'] * 2).astype('category')
cat_s.cat.codes==cat_s.array.codes
cat_s.cat.categories==cat_s.array.categories

In [ ]:
#use cat.set_categories() to change the possible categories
actual_categories = list(cat_s.cat.categories)+['e']
cat_s2 = cat_s.cat.set_categories(actual_categories)
cat_s2.value_counts()

<i>cat.remove_unused_categories()</i> trims unobserved categories, useful after filtering data

In [ ]:
cat_s3 = cat_s[cat_s.isin(['a', 'b'])]
cat_s3.cat.remove_unused_categories()

<table>
<tr><th>Method</th>
<th>Description</th></tr>
<tr><td>add_categories</td>
<td>Append new (unused) categories at end of existing categories</td></tr>
<tr><td>as_ordered</td>
<td>Make categories ordered</td></tr>
<tr><td>as_unordered</td>
<td>Make categories unordered</td></tr>
<tr><td>remove_categories</td>
<td>Remove categories, setting any removed values to null</td></tr>
<tr><td>remove_unused_categories</td>
<td>Remove any category values that do not appear in the data</td></tr>
<tr><td>rename_categories</td>
<td>Replace categories with indicated set of new category names;<br>cannot change the number of categories</td></tr>
<tr><td>reorder_categories</td>
<td>Behaves like rename_categories, but can also change the result<br>to have ordered categories</td></tr>
<tr><td>set_categories</td>
<td>Replace the categories with the indicated set of new categories;<br>can add or remove categories</td></tr>
</table>